# Data Prep

In [ ]:
import zipfile
from pathlib import Path
import rasterio
from rasterio.merge import merge
from rasterio.warp import transform_bounds
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling

import shutil
import subprocess
import tempfile
import geopandas as gpd
import rioxarray as rxr
import xarray as xr


# Base path for all data
base_path = Path("/home/xx/data/LEON_P5_BII")

# Merging (zip)filed tifs

## Merging all .tif files within ZIP

In [ ]:
##____Configuration____## 

country = "dnk"
year = 2022
dataset = "bare_before"
folder = "Bare"

# Paths #
input_dir = base_path / "EO_data_raw" / folder / f"{dataset}_{country}_{year}"
output_dir = base_path / "EO_data_prep" / folder
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {input_dir}")
print(f"Directory exists: {input_dir.exists()}")

# Find ZIP files #
zip_files = list(input_dir.glob("*.zip"))
print(f"Found {len(zip_files)} zip files")

if not zip_files:
    raise FileNotFoundError(f"No zip files found in {input_dir}")

# Extract TIFF files from ZIPs to temporary directory #
temp_dir = Path("temp")
temp_dir.mkdir(exist_ok=True)

tif_files = []

for zf in zip_files:
    print(f"Processing: {zf.name}")
    with zipfile.ZipFile(zf) as z:
        tif_matches = [n for n in z.namelist() if n.endswith(".tif")]
        if not tif_matches:
            print(f"  WARNING: No .tif files in {zf.name}")
            continue

        tif_name = tif_matches[0]
        z.extract(tif_name, temp_dir)
        tif_files.append(temp_dir / tif_name)

print(f"\nTotal .tif files extracted: {len(tif_files)}")

# Merge #
src_files = []
try:
    src_files = [rasterio.open(f) for f in tif_files]
    mosaic, out_transform = merge(src_files)

    output_file = output_dir / f"{dataset}_{country}_{year}.tif"

    with rasterio.open(
        output_file,
        "w",
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        count=mosaic.shape[0],
        dtype=mosaic.dtype,
        crs=src_files[0].crs,
        transform=out_transform,
        compress="LZW",
        tiled=True,
    ) as dst:
        dst.write(mosaic)

    print(f"Merged file saved to: {output_file}")

finally:
    for src in src_files:
        src.close()
    shutil.rmtree(temp_dir)

## Only merge zipfiles within Country bounding box

In [ ]:
##____Configuration____## 

# Bounding box (choose country)
bbox_wgs84 = (8.076389, 54.559029, 15.193056, 57.751526)  # Denmark
# bbox_wgs84 = (3.360782, 50.723492, 7.227095, 53.554585)  # Netherlands

country = "dnk"
year = 2021

dataset = "SWF"                     
folder = "Small_Woody_Features"

tif_token = "CLMS_HRLSLF_SWF_"       # e.g."WVM_"

# Paths #
input_dir = base_path / "EO_data_raw" / folder / f"{dataset}_{year}"
output_dir = base_path / "EO_data_prep" / folder
output_dir.mkdir(parents=True, exist_ok=True)

temp_dir = Path("temp")
temp_dir.mkdir(exist_ok=True)

zip_pattern = f"{dataset}_{year}_*.zip"


# Find ZIP files #
zip_files = list(input_dir.glob(zip_pattern))
if not zip_files:
    raise FileNotFoundError(f"No ZIP files found with pattern {zip_pattern}")

print(f"Found {len(zip_files)} zip files")


# Get CRS from first tile #
sample_zip = zip_files[0]
with zipfile.ZipFile(sample_zip) as z:
    tif_name = next(
        n for n in z.namelist()
        if n.endswith(".tif") and tif_token in n
    )
    z.extract(tif_name, temp_dir)

with rasterio.open(temp_dir / tif_name) as src:
    tile_crs = src.crs

shutil.rmtree(temp_dir)
temp_dir.mkdir()

bbox_proj = transform_bounds("EPSG:4326", tile_crs, *bbox_wgs84)
print(f"{country.upper()} bbox in {tile_crs}: {bbox_proj}")


# Extract intersecting tiles #
tif_files = []

for zf in zip_files:
    with zipfile.ZipFile(zf) as z:
        matches = [
            n for n in z.namelist()
            if n.endswith(".tif") and tif_token in n
        ]
        if not matches:
            continue

        tif_name = matches[0]
        z.extract(tif_name, temp_dir)
        tif_path = temp_dir / tif_name

        with rasterio.open(tif_path) as src:
            left, bottom, right, top = src.bounds
            if (
                right < bbox_proj[0]
                or left > bbox_proj[2]
                or top < bbox_proj[1]
                or bottom > bbox_proj[3]
            ):
                tif_path.unlink(missing_ok=True)
            else:
                tif_files.append(str(tif_path))

if not tif_files:
    raise FileNotFoundError("No intersecting tiles found")

print(f"Selected {len(tif_files)} tiles")

# Build VRT & Merge #
vrt_file = output_dir / f"{country}_mosaic.vrt"
merged_file = output_dir / f"{dataset}_{country}_{year}_merged.tif"

subprocess.run(["gdalbuildvrt", str(vrt_file), *tif_files], check=True)
subprocess.run(
    [
        "gdal_translate",
        str(vrt_file),
        str(merged_file),
        "-co", "COMPRESS=LZW",
        "-co", "TILED=YES",
    ],
    check=True,
)

shutil.rmtree(temp_dir)
print(f"✅ Merged file saved to: {merged_file}")

## Merging multiple countries and years in folder (not zipped)
Global Human Settlement Layer

In [ ]:
##____Configuration____## 

dataset = "ghsl_pop"
folder = "GHSL"

countries_years = {
    "ndl": [2015, 2020, 2025],
    "cnk": [2015, 2020, 2025],
    "uk":  [2015, 2025],
}

# Paths #
raw_base = base_path / "EO_data_raw" / folder
output_base = base_path / "EO_data_prep" / folder
output_base.mkdir(parents=True, exist_ok=True)

# Process each country-year combination #
for country, years in countries_years.items():
    for year in years:
        input_dir = raw_base / str(year)
        if not input_dir.exists():
            continue

        tif_files = list(input_dir.rglob("*.tif"))
        if not tif_files:
            continue

        src_files = [rasterio.open(f) for f in tif_files]
        mosaic, out_transform = merge(src_files)

        output_file = output_base / f"{dataset}_{country}_{year}_merged.tif"

        with rasterio.open(
            output_file,
            "w",
            driver="GTiff",
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            count=mosaic.shape[0],
            dtype=mosaic.dtype,
            crs=src_files[0].crs,
            transform=out_transform,
            compress="LZW",
            tiled=True,
        ) as dst:
            dst.write(mosaic)

        for src in src_files:
            src.close()

# Clip to Country boundaries

In [ ]:
# Load country gpkg files
uk = gpd.read_file(base_path/ "Country_Boundaries/gadm41_GBR_adm2_filtered4_3035.gpkg")
dnk = gpd.read_file(base_path / "Country_Boundaries/gadm41_dnk_adm_0_3035.gpkg")
nld = gpd.read_file(base_path / "Country_Boundaries/gadm41_nld_adm_0_3035.gpkg")

#### Batch Clip

In [ ]:
##____Configuration____## 

aoi = nld

country = "nld"
years = [2018, 2020, 2023]   # ← batch over years here

dataset = "bare_after"
folder = "Bare"

# Clip #
for year in years:
    print(f"\nProcessing year: {year}")

    tif_file = base_path / "EO_data_prep" / folder / f"{dataset}_{country}_{year}.tif"
    output_file = base_path / "EO_data_prep" / folder / f"{dataset}_{country}_{year}_clip.tif"

    with rasterio.open(tif_file) as src:
        # Minimal CRS check
        aoi_use = aoi
        if getattr(aoi, "crs", None) and aoi.crs != src.crs:
            aoi_use = aoi.to_crs(src.crs)

        geom = [aoi_use.geometry.unary_union]
        clipped, transform = mask(src, geom, crop=True, all_touched=True)

        with rasterio.open(
            output_file,
            "w",
            driver="GTiff",
            height=clipped.shape[1],
            width=clipped.shape[2],
            count=src.count,
            dtype=clipped.dtype,
            crs=src.crs,
            transform=transform,
            compress="LZW",
            tiled=True,
        ) as dst:
            dst.write(clipped)

    print(f"Saved: {output_file}")

#### Entire folder batch + reprojection e.g. Nightlights

In [ ]:
import rasterio
from rasterio.mask import mask
from pathlib import Path

### For files ###

##___Config___##
aoi = nld   # Using country boundaries for clipping

country = "nld"
year = 2023
dataset = "bare_after"
folder = "Bare" 

# Path can be a file or folder
tif_file = base_path / "EO_data_prep" / folder / f"{dataset}_{country}_{year}.tif"

# Check if it's a file or folder
if tif_path.is_file():
    tif_files = [tif_path]
elif tif_path.is_dir():
    tif_files = list(tif_path.rglob("*.tif"))
else:
    raise FileNotFoundError(f"Path not found: {tif_path}")

print(f"Found {len(tif_files)} .tif file(s)")

# Load and clip each tif
for tif_file in tif_files:
    print(f"\nProcessing: {tif_file.name}")
    
    with rasterio.open(tif_file) as src:
        # Clip with AOI
        aoi_geom = [aoi.geometry.unary_union]
        aoi_clipped, aoi_transform = mask(src, aoi_geom, crop=True, all_touched=True)
        
        # Generate output filename
        output_name = tif_file.stem + "_clip.tif"
        output_path = tif_file.parent / output_name
        
        # Save clipped result
        with rasterio.open(output_path, 'w',
                          driver='GTiff',
                          height=aoi_clipped.shape[1],
                          width=aoi_clipped.shape[2],
                          count=src.count,
                          dtype=aoi_clipped.dtype,
                          crs=src.crs,
                          transform=aoi_transform,
                          compress='LZW') as dst:
            dst.write(aoi_clipped)
        
        print(f"Saved: {output_path}")

### For MODIS

In [ ]:
# Multiprocess clipping with GDAL

# --- Config ---
aoi = dnk  # Already loaded GeoDataFrame
country = "dnk"
year = 2021
dataset = "NPP"
folder = "Modis_NPP"

# Input raster
tif_path = base_path / "EO_data_prep" / folder / f"{dataset}_{country}_{year}.tif"
output_path = base_path / "EO_data_prep" / folder / f"{dataset}_{country}_{year}_clip.tif"

# Check input file exists
if not tif_path.exists():
    raise FileNotFoundError(f"Input raster not found: {tif_path}")

# --- Ensure AOI is in EPSG:3035 ---
target_crs = "EPSG:3035"
if aoi.crs != target_crs:
    print(f"Reprojecting AOI from {aoi.crs} to {target_crs}")
    aoi = aoi.to_crs(target_crs)
else:
    print(f"AOI already in {target_crs}")

# --- Save AOI to temporary GPKG for GDAL ---
with tempfile.TemporaryDirectory() as tmpdir:
    tmp_gpkg = Path(tmpdir) / "aoi.gpkg"
    aoi.to_file(tmp_gpkg, driver="GPKG")

    # --- GDAL command with reprojection ---
    cmd = [
        "gdalwarp",
        "-cutline", str(tmp_gpkg),
        "-crop_to_cutline",
        "-wo", "CUTLINE_ALL_TOUCHED=TRUE",          #  Include pixels that touch boundary
        "-t_srs", target_crs,
        "-tr", "300", "300",
        "-r", "bilinear",                #  Bilinear for continuous data
        "-of", "GTiff",
        "-co", "COMPRESS=LZW",
        "-co", "TILED=YES",
        "-multi",
        "-wo", "NUM_THREADS=ALL_CPUS",
        str(tif_path),
        str(output_path)
    ]

    print(f"Clipping and reprojecting {tif_path.name}...")
    subprocess.run(cmd, check=True)

print(f"✅ Clipped raster saved to: {output_path}")

# --- Verify output CRS ---
with rasterio.open(output_path) as src:
    print(f"Output CRS: {src.crs}")
    print(f"Output shape: {src.shape}")
    print(f"Output bounds: {src.bounds}")
    print(f"Output resolution: {src.res}")

### WorldClim Temp average + Reproject + Clip

In [ ]:
# Prepare WorldClim data. Calc mean annual temperature average from monthly rasters.

# --- Config --- #
aoi = nld  
country = "nld"
months = range(1, 13)  # 1-12
dataset = "wc2.1_2.5m_tavg"
folder = "Temp_avg"

input_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_raw/{folder}")
output_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_mean_annual.tif")
output_path.parent.mkdir(parents=True, exist_ok=True)

# --- Process --- #
# Load all monthly rasters
monthly_rasters = []
for month in months:
    tif_path = input_dir / f"{dataset}_{month:02d}.tif"
    raster = rxr.open_rasterio(tif_path, masked=True)
    monthly_rasters.append(raster)

# Calculate mean annual temperature
mean_temp = xr.concat(monthly_rasters, dim='month').mean(dim='month')

# Reproject to EPSG:3035
mean_temp_reprojected = mean_temp.rio.reproject("EPSG:3035")

# Clip to country boundary
mean_temp_clipped = mean_temp_reprojected.rio.clip(aoi.geometry, aoi.crs, all_touched=True)

# Save
mean_temp_clipped.rio.to_raster(output_path, compress='lzw')

print(f"Saved: {output_path}")